# VA-π: Fine-tune GRPO trên checkpoint pretrained (CIFAR-100 & STL-10)

**Nội dung notebook:**
1. Cài đặt môi trường, clone repo `Lil-Shake/VA-Pi`, tải checkpoint pretrained chính thức
2. Chuẩn bị 2 dataset mới: CIFAR-100 (100 lớp) và STL-10 (10 lớp) theo định dạng ImageFolder
3. "Patch" checkpoint pretrained để tương thích số lớp mới (giữ nguyên backbone Transformer)
4. Fine-tune GRPO (VA-π) trên từng dataset, dùng script DDP có sẵn trong repo — phù hợp Kaggle 2×T4
5. Đánh giá FID/IS **trước** và **sau** fine-tune cho cả 2 dataset

**Model dùng:** `GPT-L` (343M) + tokenizer `vq_ds16_c2i` (chính thức, `FoundationVision/LlamaGen`
trên HuggingFace). Không dùng GPT-XL/XXL vì script GRPO gốc cần 8 GPU/FSDP — sẽ OOM trên 2×T4 16GB.
Nếu vẫn OOM với GPT-L, đổi sang `GPT-B` (111M) — hướng dẫn đổi ở Cell cấu hình.

**Lưu ý về thời gian:** Kaggle free tier giới hạn phiên GPU ~9-12h và ~30h GPU T4/tuần.
Notebook này lưu checkpoint định kỳ để có thể resume ở phiên sau nếu cần.


## 1. Cài đặt môi trường & tải checkpoint pretrained

In [1]:
# Kiểm tra GPU
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv


name, memory.total [MiB], memory.used [MiB]
Tesla T4, 15360 MiB, 0 MiB
Tesla T4, 15360 MiB, 0 MiB


In [2]:
import os

WORK = "/kaggle/working"
REPO = f"{WORK}/VA-Pi"
PRETRAINED = f"{WORK}/pretrained"
DATA_ROOT = f"{WORK}/data"
RUNS = f"{WORK}/runs"

for d in [PRETRAINED, DATA_ROOT, RUNS]:
    os.makedirs(d, exist_ok=True)

if not os.path.isdir(REPO):
    !git clone --depth 1 https://github.com/Lil-Shake/VA-Pi.git {REPO}
else:
    print("Repo đã tồn tại, bỏ qua clone.")


Cloning into '/kaggle/working/VA-Pi'...
remote: Enumerating objects: 184, done.
remote: Counting objects: 100% (184/184), done.
remote: Compressing objects: 100% (140/140), done.
remote: Total 184 (delta 44), reused 171 (delta 39), pack-reused 0 (from 0)
Receiving objects: 100% (184/184), 6.87 MiB | 41.38 MiB/s, done.
Resolving deltas: 100% (44/44), done.


In [3]:
%cd {REPO}/LlamaGen
!pip install -r requirements.txt --break-system-packages -q
!pip install torchmetrics torch-fidelity --break-system-packages -q
!pip install torch-fidelity --break-system-packages -q


/kaggle/working/VA-Pi/LlamaGen
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.3/266.3 kB 11.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.6/111.6 kB 7.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.1/110.1 kB 8.2 MB/

In [4]:
# Tải checkpoint pretrained chính thức (VQ tokenizer + AR model GPT-L, ImageNet C2I 256x256)
import os

vq_path = f"{PRETRAINED}/vq_ds16_c2i.pt"
gpt_path = f"{PRETRAINED}/c2i_L_256.pt"

if not os.path.exists(vq_path):
    !wget -q --show-progress -O {vq_path} https://huggingface.co/FoundationVision/LlamaGen/resolve/main/vq_ds16_c2i.pt
if not os.path.exists(gpt_path):
    !wget -q --show-progress -O {gpt_path} https://huggingface.co/FoundationVision/LlamaGen/resolve/main/c2i_L_256.pt

print("VQ:", os.path.getsize(vq_path) / 1e6, "MB")
print("GPT-L:", os.path.getsize(gpt_path) / 1e6, "MB")


/kaggle/working/pre 100%[===================>] 274.58M  73.8MB/s    in 3.7s    
/kaggle/working/pre 100%[===================>]   1.28G  75.5MB/s    in 28s     
VQ: 287.920306 MB
GPT-L: 1371.712314 MB


> **Đổi sang GPT-B nếu OOM:** thay `c2i_L_256.pt` -> `c2i_B_256.pt` trong URL wget ở trên,
> và sau này đổi toàn bộ `GPT_MODEL = "GPT-L"` -> `"GPT-B"` ở Cell cấu hình chung bên dưới.

## 2. Cấu hình chung

In [5]:
GPT_MODEL = "GPT-L"          # đổi thành "GPT-B" nếu bị OOM (111M, nhẹ hơn nhiều)
IMAGE_SIZE = 256              # phải khớp với tokenizer vq_ds16_c2i (train ở 256)
LIMIT_PER_CLASS = 200         # giới hạn số ảnh/lớp để vừa ngân sách giờ Kaggle; đặt None để dùng full data
EPOCHS = 1                    # bắt đầu nhỏ để đo tốc độ thực tế trước khi tăng lên 10+

DATASETS = {
    "cifar100": {"num_classes": 100},
    "stl10":    {"num_classes": 10},
}


## 3. Chuẩn bị dataset (CIFAR-100 & STL-10 -> ImageFolder)

In [6]:
"""
Tải CIFAR-100 và STL-10 qua torchvision, resize (upsample) lên IMAGE_SIZE,
và ghi ra cấu trúc thư mục kiểu ImageFolder mà VA-Pi/LlamaGen/dataset/imagenet.py
(build_imagenet -> torchvision.datasets.ImageFolder) cần: mỗi lớp = 1 thư mục con.
"""
import torchvision
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm


def dump_imagefolder(dataset, class_names, out_dir: Path, image_size: int, limit_per_class=None):
    out_dir.mkdir(parents=True, exist_ok=True)
    counters = {}
    for idx in tqdm(range(len(dataset)), desc=f"-> {out_dir}"):
        img, label = dataset[idx]
        cname = class_names[label] if class_names else str(label)
        counters[cname] = counters.get(cname, 0) + 1
        if limit_per_class is not None and counters[cname] > limit_per_class:
            continue
        cdir = out_dir / cname
        cdir.mkdir(parents=True, exist_ok=True)
        if not isinstance(img, Image.Image):
            img = torchvision.transforms.functional.to_pil_image(img)
        img = img.convert("RGB").resize((image_size, image_size), Image.BICUBIC)
        img.save(cdir / f"{counters[cname]:06d}.jpg", quality=95)


In [7]:
raw_dir = Path(DATA_ROOT) / "_raw"
raw_dir.mkdir(parents=True, exist_ok=True)

print("== CIFAR-100 ==")
c100_train = torchvision.datasets.CIFAR100(str(raw_dir), train=True, download=True)
c100_test  = torchvision.datasets.CIFAR100(str(raw_dir), train=False, download=True)
dump_imagefolder(c100_train, c100_train.classes, Path(DATA_ROOT) / "cifar100" / "train", IMAGE_SIZE, LIMIT_PER_CLASS)
dump_imagefolder(c100_test,  c100_train.classes, Path(DATA_ROOT) / "cifar100" / "val",   IMAGE_SIZE, LIMIT_PER_CLASS)


== CIFAR-100 ==


100%|██████████| 169M/169M [1:13:29<00:00, 38.3kB/s]


-> /kaggle/working/data/cifar100/train:   0%|          | 0/50000 [00:00<?, ?it/s]

-> /kaggle/working/data/cifar100/val:   0%|          | 0/10000 [00:00<?, ?it/s]

In [8]:
print("== STL-10 ==")
stl_train = torchvision.datasets.STL10(str(raw_dir), split="train", download=True)
stl_test  = torchvision.datasets.STL10(str(raw_dir), split="test", download=True)
dump_imagefolder(stl_train, stl_train.classes, Path(DATA_ROOT) / "stl10" / "train", IMAGE_SIZE, LIMIT_PER_CLASS)
dump_imagefolder(stl_test,  stl_train.classes, Path(DATA_ROOT) / "stl10" / "val",   IMAGE_SIZE, LIMIT_PER_CLASS)


== STL-10 ==


100%|██████████| 2.64G/2.64G [01:05<00:00, 40.5MB/s]


-> /kaggle/working/data/stl10/train:   0%|          | 0/5000 [00:00<?, ?it/s]

-> /kaggle/working/data/stl10/val:   0%|          | 0/8000 [00:00<?, ?it/s]

In [9]:
for name in DATASETS:
    p = Path(DATA_ROOT) / name / "train"
    print(name, "->", len(list(p.iterdir())), "lớp,",
          sum(len(list(c.iterdir())) for c in p.iterdir()), "ảnh train")


cifar100 -> 100 lớp, 20000 ảnh train
stl10 -> 10 lớp, 2000 ảnh train


## 4. Patch checkpoint pretrained cho từng số lớp mới

Checkpoint gốc train trên ImageNet (1000 lớp) nên bảng `cls_embedding.embedding_table`
có shape `[1001, dim]`. CIFAR-100 = 100 lớp, STL-10 = 10 lớp -> phải xoá riêng các key
`cls_embedding.*` trước khi nạp, để tránh lỗi *size mismatch*. Phần backbone Transformer
(attention, MLP, RMSNorm...) giữ nguyên 100% trọng số pretrained.

In [10]:
import torch

def strip_class_embedding(in_ckpt: str, out_ckpt: str):
    ckpt = torch.load(in_ckpt, map_location="cpu")
    if isinstance(ckpt, dict) and "model" in ckpt:
        state, wrapper_key = ckpt["model"], "model"
    else:
        state, wrapper_key = ckpt, None

    removed = [k for k in state.keys() if k.startswith("cls_embedding")]
    print(f"[{out_ckpt}] loại bỏ {len(removed)} key:", removed)
    for k in removed:
        del state[k]

    if wrapper_key:
        ckpt[wrapper_key] = state
        torch.save(ckpt, out_ckpt)
    else:
        torch.save(state, out_ckpt)

for name in DATASETS:
    strip_class_embedding(
        gpt_path,
        f"{PRETRAINED}/{GPT_MODEL}_stripped_{name}.pt",
    )


[/kaggle/working/pretrained/GPT-L_stripped_cifar100.pt] loại bỏ 1 key: ['cls_embedding.embedding_table.weight']
[/kaggle/working/pretrained/GPT-L_stripped_stl10.pt] loại bỏ 1 key: ['cls_embedding.embedding_table.weight']


## 5. Fine-tune GRPO (VA-π) trên từng dataset

Dùng `train_c2i_ddp_grpo.py` có sẵn trong repo (bản DDP, không bắt buộc FSDP) —
phù hợp 2 GPU T4 hơn bản mặc định (`train_c2i_grpo.py`, thiết kế cho FSDP nhiều GPU lớn).

**Lưu ý phần cứng:** T4 là kiến trúc Turing, không có bf16 tensor-core -> luôn dùng
`--mixed-precision fp16` (khác mặc định `bf16` của repo, vốn thiết kế cho A100/H100).

In [11]:
import subprocess

def train_one(dataset_name: str, num_classes: int):
    data_path = f"{DATA_ROOT}/{dataset_name}/train"
    gpt_ckpt = f"{PRETRAINED}/{GPT_MODEL}_stripped_{dataset_name}.pt"
    out_dir = f"{RUNS}/{dataset_name}"
    os.makedirs(f"{out_dir}/ckpt", exist_ok=True)

    cmd = [
        "torchrun",
        "--nnodes=1", "--nproc_per_node=2", "--node_rank=0",
        "--master_addr=127.0.0.1", "--master_port=12355",
        "autoregressive/train/train_c2i_ddp_grpo.py",
        "--vq-ckpt", vq_path,
        "--gpt-ckpt", gpt_ckpt,
        "--gpt-model", GPT_MODEL,
        "--data-path", data_path,
        "--dataset", "imagenet",
        "--data-loader", "builtin",
        "--num-classes", str(num_classes),
        "--image-size", str(IMAGE_SIZE), "--downsample-size", "16",
        "--data-parallel", "sdp",
        "--mixed-precision", "fp16",
        "--sample-batch-size", "8",
        "--train-batch-size", "4",
        "--gradient-accumulation-steps", "8",
        "--num-generations", "4",
        "--temperature", "1.0", "--top-k", "0", "--top-p", "1.0",
        "--epochs", str(EPOCHS),
        "--ckpt-every", "200", "--ckpt-dir", f"{out_dir}/ckpt",
        "--lr", "1e-6", "--weight-decay", "1e-4", "--beta1", "0.9", "--beta2", "0.999",
        "--max-grad-norm", "1.0", "--clip-range", "0.2", "--adv-clip-max", "5.0",
        "--reward-rec-weight", "1.0", "--reward-perceptual-weight", "1.0", "--aux-ce-weight", "0.1",
        "--use-kl-loss", "--kl-coef", "0.01",
        "--sample-model-mode", "eval",
        "--log-interval", "10", "--log-image-every", "100",
    ]
    log_path = f"{out_dir}/train.log"
    print(f"===== Bắt đầu fine-tune: {dataset_name} (num_classes={num_classes}) =====")
    with open(log_path, "w") as f:
        proc = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT)
    print(f"Kết thúc với return code {proc.returncode}. Log: {log_path}")
    return proc.returncode


In [12]:
# --- Chạy fine-tune cho CIFAR-100 ---
rc = train_one("cifar100", DATASETS["cifar100"]["num_classes"])
!tail -n 40 {RUNS}/cifar100/train.log


===== Bắt đầu fine-tune: cifar100 (num_classes=100) =====
Kết thúc với return code 1. Log: /kaggle/working/runs/cifar100/train.log
ModuleNotFoundError: No module named 'autoregressive'
E0709 15:58:00.987000 117 torch/distributed/elastic/multiprocessing/api.py:984] failed (exitcode: 1) local_rank: 0 (pid: 123) of binary: /usr/bin/python3
Traceback (most recent call last):
  File "/usr/local/bin/torchrun", line 10, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/distributed/elastic/multiprocessing/errors/__init__.py", line 362, in wrapper
    return f(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/distributed/run.py", line 991, in main
    run(args)
  File "/usr/local/lib/python3.12/dist-packages/torch/distributed/run.py", line 982, in run
    elastic_launch(
  File "/usr/local/lib/python3.12/dist-packages/torch/distributed/launcher/api.py", line 170, in __call__
    return la

In [13]:
# --- Chạy fine-tune cho STL-10 ---
rc = train_one("stl10", DATASETS["stl10"]["num_classes"])
!tail -n 40 {RUNS}/stl10/train.log


===== Bắt đầu fine-tune: stl10 (num_classes=10) =====
Kết thúc với return code 1. Log: /kaggle/working/runs/stl10/train.log
ModuleNotFoundError: No module named 'autoregressive'
E0709 15:58:09.734000 128 torch/distributed/elastic/multiprocessing/api.py:984] failed (exitcode: 1) local_rank: 0 (pid: 134) of binary: /usr/bin/python3
Traceback (most recent call last):
  File "/usr/local/bin/torchrun", line 10, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/distributed/elastic/multiprocessing/errors/__init__.py", line 362, in wrapper
    return f(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/distributed/run.py", line 991, in main
    run(args)
  File "/usr/local/lib/python3.12/dist-packages/torch/distributed/run.py", line 982, in run
    elastic_launch(
  File "/usr/local/lib/python3.12/dist-packages/torch/distributed/launcher/api.py", line 170, in __call__
    return launch_ag

> **Nếu phiên Kaggle sắp hết giờ / bị ngắt:** lưu thư mục `/kaggle/working/runs/`
> thành 1 Kaggle Dataset version. Ở phiên sau, load lại và thay `gpt_ckpt` trong
> `train_one()` bằng checkpoint gần nhất trong `runs/<dataset>/ckpt/` để resume.
>
> **Nếu OOM ngay từ bước đầu:** giảm `--num-generations` xuống 2, `--sample-batch-size`
> xuống 4, hoặc đổi `GPT_MODEL = "GPT-B"` ở Cell cấu hình và chạy lại từ Cell patch checkpoint.

## 6. Đánh giá FID/IS trước & sau fine-tune

So sánh:
- **Trước**: checkpoint pretrained đã patch class-embedding (random-init embedding, backbone gốc), CHƯA GRPO
- **Sau**: checkpoint sau khi fine-tune GRPO

Lặp lại cho cả CIFAR-100 và STL-10 -> 4 lần đánh giá tổng cộng.

In [14]:
import sys
sys.path.insert(0, f"{REPO}/LlamaGen")

import torch
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.inception import InceptionScore

from tokenizer.tokenizer_image.vq_model import VQ_models
from autoregressive.models.gpt import GPT_models
from autoregressive.models.generate import generate

device = "cuda" if torch.cuda.is_available() else "cpu"

_vq_cache = {}
def get_vq():
    if "vq" not in _vq_cache:
        vq = VQ_models["VQ-16"](codebook_size=16384, codebook_embed_dim=8)
        ckpt = torch.load(vq_path, map_location="cpu")
        vq.load_state_dict(ckpt["model"] if "model" in ckpt else ckpt)
        _vq_cache["vq"] = vq.to(device).eval()
    return _vq_cache["vq"]


def load_gpt(gpt_ckpt_path: str, num_classes: int):
    latent_size = IMAGE_SIZE // 16
    gpt = GPT_models[GPT_MODEL](
        vocab_size=16384, block_size=latent_size ** 2,
        num_classes=num_classes, cls_token_num=1, model_type="c2i",
    )
    ckpt = torch.load(gpt_ckpt_path, map_location="cpu")
    state = ckpt["model"] if "model" in ckpt else ckpt
    gpt.load_state_dict(state, strict=False)
    return gpt.to(device).eval(), latent_size


@torch.no_grad()
def generate_batch(vq, gpt, labels, latent_size):
    idx = generate(gpt, labels.to(device), latent_size ** 2,
                    cfg_scale=1.0, temperature=1.0, top_k=0, top_p=1.0)
    imgs = vq.decode_code(idx, shape=(labels.shape[0], 8, latent_size, latent_size))
    imgs = torch.clamp(127.5 * imgs + 128.0, 0, 255).to(torch.uint8)
    return imgs


def eval_fid_is(gpt_ckpt_path: str, num_classes: int, real_dir: str, n_samples: int, batch_size: int, tag: str):
    vq = get_vq()
    gpt, latent_size = load_gpt(gpt_ckpt_path, num_classes)

    fid = FrechetInceptionDistance(feature=2048, normalize=False).to(device)
    iscore = InceptionScore(normalize=False).to(device)

    tfm = transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), transforms.PILToTensor()])
    real_loader = DataLoader(ImageFolder(real_dir, transform=tfm), batch_size=batch_size,
                              shuffle=True, num_workers=2)

    n_real = 0
    for imgs, _ in real_loader:
        fid.update(imgs.to(device), real=True)
        n_real += imgs.shape[0]
        if n_real >= n_samples:
            break

    n_gen = 0
    while n_gen < n_samples:
        bs = min(batch_size, n_samples - n_gen)
        labels = torch.randint(0, num_classes, (bs,))
        fake = generate_batch(vq, gpt, labels, latent_size)
        fid.update(fake, real=False)
        iscore.update(fake)
        n_gen += bs

    fid_score = fid.compute().item()
    is_mean, is_std = iscore.compute()
    result = {"tag": tag, "fid": fid_score, "is_mean": is_mean.item(), "is_std": is_std.item(),
              "n_real": n_real, "n_gen": n_gen}
    print(result)
    del gpt
    torch.cuda.empty_cache()
    return result


In [15]:
N_EVAL_SAMPLES = 500   # tăng lên (vd 2000-5000) nếu muốn FID ổn định hơn, nhưng sẽ chậm hơn nhiều
EVAL_BATCH = 16

results = []
for name, cfg in DATASETS.items():
    real_dir = f"{DATA_ROOT}/{name}/val"
    before_ckpt = f"{PRETRAINED}/{GPT_MODEL}_stripped_{name}.pt"
    after_ckpt_dir = f"{RUNS}/{name}/ckpt"
    after_candidates = sorted(Path(after_ckpt_dir).glob("*.pt")) if os.path.isdir(after_ckpt_dir) else []
    after_ckpt = str(after_candidates[-1]) if after_candidates else None

    results.append(eval_fid_is(before_ckpt, cfg["num_classes"], real_dir,
                                N_EVAL_SAMPLES, EVAL_BATCH, tag=f"{name}_before"))
    if after_ckpt:
        results.append(eval_fid_is(after_ckpt, cfg["num_classes"], real_dir,
                                    N_EVAL_SAMPLES, EVAL_BATCH, tag=f"{name}_after"))
    else:
        print(f"[{name}] Chưa tìm thấy checkpoint sau fine-tune trong {after_ckpt_dir}, bỏ qua đánh giá 'after'.")


Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/weights-inception-2015-12-05-6726825d.pth
100%|██████████| 91.2M/91.2M [00:00<00:00, 351MB/s]
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


{'tag': 'cifar100_before', 'fid': 176.52183532714844, 'is_mean': 10.413039207458496, 'is_std': 0.8456565141677856, 'n_real': 512, 'n_gen': 500}
[cifar100] Chưa tìm thấy checkpoint sau fine-tune trong /kaggle/working/runs/cifar100/ckpt, bỏ qua đánh giá 'after'.
{'tag': 'stl10_before', 'fid': 192.2083282470703, 'is_mean': 8.997848510742188, 'is_std': 1.090213656425476, 'n_real': 512, 'n_gen': 500}
[stl10] Chưa tìm thấy checkpoint sau fine-tune trong /kaggle/working/runs/stl10/ckpt, bỏ qua đánh giá 'after'.


In [16]:
import pandas as pd
df = pd.DataFrame(results)
df


,tag,fid,is_mean,is_std,n_real,n_gen
0,cifar100_before,176.521835,10.413039,0.845657,512,500
1,stl10_before,192.208328,8.997849,1.090214,512,500


## 7. Kết luận

So sánh cột `fid`/`is_mean` giữa `<dataset>_before` và `<dataset>_after` trong bảng trên
chính là bằng chứng định lượng cho việc pretrained model (VA-π trên LlamaGen-{GPT_MODEL})
tổng quát hoá được sang 2 domain mới ngoài ImageNet: CIFAR-100 và STL-10.

**Hạn chế cần nêu trong báo cáo:** ảnh gốc CIFAR-100 (32×32) và STL-10 (96×96) nhỏ hơn
nhiều so với 256×256 mà tokenizer `vq_ds16_c2i` kỳ vọng, nên sau upsample ảnh sẽ hơi mờ,
ảnh hưởng đến rFID nền tảng trước cả khi fine-tune GRPO — đây là hạn chế cố hữu khi tái sử
dụng tokenizer pretrain trên domain khác thay vì train VQ-VAE riêng cho từng dataset.